# 6주차 ① 옵티마이저와 학습률 스케줄러 — 실습 1~3  〔교수용 정답본〕

> ⚠️ **이 파일은 교수용입니다.** 학생에게는 `08_optimizers_blank.ipynb` 를 배포하세요.
> 빈칸 7곳 · 13줄이 모두 채워져 있습니다.

---

## 실습 1 — SGD vs Momentum vs Adam ★ (12분)

**비교 실험은 하나만 바꿔야 성립합니다.**
모델·시드·데이터·epoch 를 전부 같게 두고 **옵티마이저만** 바꿉니다.
6주차 과제의 채점 기준이 정확히 이것입니다.

| 옵티마이저 | 무엇을 해결했나 | 기본 lr |
|---|---|---|
| **SGD** | (출발점) — 4주차에 손으로 쓴 그것 | `0.1` 대 |
| **+ Momentum** | 골짜기에서의 **지그재그** | `0.1` 대 |
| **Adam** | 축마다 다른 기울기 크기 → **파라미터별 학습률** | `1e-3` 대 ★ |

> ⚠️ **학습률의 자릿수가 다릅니다.** Adam 에 `0.1` 을 주면 대개 발산합니다.
> **옵티마이저를 바꾸면 lr 도 함께 바꿔야** 합니다 — 실습 2에서 직접 봅니다.

In [ ]:
# 셀 1 — 공통 준비
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("사용할 장치 :", device)

ROOT = "data"
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.2860,), (0.3530,))])

train_full = datasets.FashionMNIST(ROOT, train=True,  download=True, transform=tf)
test_set   = datasets.FashionMNIST(ROOT, train=False, download=True, transform=tf)

N_TRAIN = 20000          # ★ 비교용이라 2만 장만 쓴다. GPU 면 60000 으로 올려도 좋다
train_set = Subset(train_full, range(N_TRAIN))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)
print("훈련", len(train_set), "장 / 테스트", len(test_set), "장 |",
      len(train_loader), "iteration/epoch")

In [ ]:
# 셀 2 — 모든 실험이 같은 출발점에서 시작하게 한다
def make_model():
    torch.manual_seed(0)                     # ★ 항상 같은 초기값에서 출발
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256), nn.ReLU(),
        nn.Linear(256, 128), nn.ReLU(),
        nn.Linear(128, 10),
    ).to(device)

m = make_model()
print("파라미터 수 :", f"{sum(p.numel() for p in m.parameters()):,} 개")

> **관찰 포인트 ★**: 5주차와 **똑같은 모델**입니다. 오늘 바꾸는 것은 **모델이 아니라 옵티마이저**입니다.
> `torch.manual_seed(0)` 이 함수 **안**에 있는 것에 주목하세요 —
> 부를 때마다 **같은 초기값**에서 시작해야 비교가 공정합니다.

In [ ]:
# 셀 3 — 학습 함수 (5주차 루프 그대로) 와 정확도 함수
def train(opt_name, epochs=5, sched=None):
    model = make_model()
    lossfn = nn.CrossEntropyLoss()

    opt = {
        "SGD":      torch.optim.SGD(model.parameters(), lr=0.1),
        "Momentum": torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9),
        "Adam":     torch.optim.Adam(model.parameters(), lr=1e-3),
    }[opt_name]

    scheduler = sched(opt) if sched else None

    hist, lrs = [], []
    for _ in range(epochs):
        model.train(); run, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossfn(model(xb), yb)              # ① 순전파 ② 손실
            opt.zero_grad(); loss.backward(); opt.step()   # ③④⑤
            run += loss.item() * xb.size(0); n += xb.size(0)

        lrs.append(opt.param_groups[0]["lr"])      # ★ 지금의 학습률
        if scheduler: scheduler.step()             # ★ epoch 끝에 한 번
        hist.append(run / n)
    return model, hist, lrs


def accuracy(model):
    model.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (model(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t


print("준비 완료 — 학습 루프는 5주차와 글자 그대로 같습니다")

In [ ]:
# 셀 4 — 세 옵티마이저를 나란히 학습시킨다  ★ 시간이 좀 걸립니다
results, accs = {}, {}

for name in ["SGD", "Momentum", "Adam"]:
    m, h, _ = train(name)
    results[name], accs[name] = h, accuracy(m)
    print(f"{name:9s} | 마지막 loss {results[name][-1]:.4f} | 테스트 정확도 {accs[name]*100:5.2f}%")

for name, h in results.items():
    plt.plot(range(1, len(h)+1), h, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("평균 loss"); plt.legend()
plt.title("옵티마이저 3종 (같은 모델·같은 시드·5 epoch)"); plt.grid(alpha=0.3)
plt.show()

> **관찰 포인트 ★** — 교수님 PC(CPU) 실측값입니다.
>
> ```
> SGD       | 마지막 loss 0.3711 | 테스트 정확도 83.46%
> Momentum  | 마지막 loss 0.3376 | 테스트 정확도 84.37%
> Adam      | 마지막 loss 0.3102 | 테스트 정확도 85.75%
> ```
>
> **세 곡선의 모양을 보세요.** 초반에 가장 빨리 내려가는 것은 **Adam** 입니다.
> 그런데 **차이가 2%p 남짓**이라는 점도 함께 보세요 —
> *"Adam 을 쓰면 확 좋아진다"* 가 아니라 **"조금 더 빨리 수렴한다"** 가 정확한 표현입니다 — 같은 loss 에 **더 적은 epoch** 로 도달한다는 뜻입니다.
>
> **Adam 이 항상 이기지는 않습니다.** 데이터를 늘리거나 오래 학습하면 순위가 바뀌기도 합니다.
> 실무 감각은 *"기본은 Adam 으로 시작하고, 성능을 향상시킬 때 SGD+Momentum 을 튜닝한다"* 입니다.

In [ ]:
# 셀 5 — 비교표를 지금 만들어 둔다 (6주차 과제에 그대로 쓴다)
print(f"{'옵티마이저':10s} {'lr':>8s} {'epoch':>6s} {'batch':>6s} "
      f"{'seed':>5s} {'최종loss':>9s} {'정확도':>8s}")
print("-" * 62)
for name, lr in [("SGD", 0.1), ("Momentum", 0.1), ("Adam", 1e-3)]:
    print(f"{name:10s} {lr:>8} {5:>6} {128:>6} {0:>5} "
          f"{results[name][-1]:>9.4f} {accs[name]*100:>7.2f}%")

> **핵심 메시지**: 이 표가 **6주차 과제의 뼈대**입니다.
>
> | 기록할 것 | 왜 |
> |---|---|
> | 옵티마이저 이름, **학습률** | lr 이 다르면 공정한 비교가 아니다 |
> | 시드 | 다시 실행했을 때 같은 결과가 나와야 한다 (3교시) |
> | epoch, batch size | 조건이 같아야 비교가 성립 |
> | 최종 손실, 테스트 정확도 | 결과 |
>
> **조건을 안 적으면 다음 주에 자기가 뭘 했는지 모릅니다.**

---

## 실습 2 — Adam 에 SGD 용 학습률을 주면 (5분)

앞에서 *"옵티마이저를 바꾸면 lr 도 바꿔야 한다"* 고 했습니다.
**안 바꾸면 무슨 일이 생기는지 직접 봅시다.**

In [ ]:
# 셀 6 — 일부러 발산시킨다
torch.manual_seed(0)
bad = make_model()

bad_opt = torch.optim.Adam(bad.parameters(), lr=0.1)      # ★ Adam 에 SGD 용 lr

lossfn = nn.CrossEntropyLoss()

bad.train()
for i, (xb, yb) in enumerate(train_loader):
    xb, yb = xb.to(device), yb.to(device)
    loss = lossfn(bad(xb), yb)
    bad_opt.zero_grad(); loss.backward(); bad_opt.step()
    if i % 40 == 0:
        print(f"iter {i:4d} | loss {loss.item():.4f}")
    if i == 150: break

print(f"\n테스트 정확도 : {accuracy(bad)*100:.2f}%   ← 실습 1 에서는 85% 대였다")

> **결과 해석 ★** — 실측값입니다.
>
> ```
> iter    0 | loss 2.3325
> iter   40 | loss 3.7749      ← 오히려 올라갔다
> iter   80 | loss 2.4623
> iter  120 | loss 2.4151
>
> 테스트 정확도 : 28.41%   ← 실습 1 의 Adam 은 85.75% 였다
> ```
>
> 손실이 **내려가기는커녕 출렁입니다.** 정확도도 85.75% → **28.41%** 로 무너졌습니다.
> **`lr=1e-3` 이었으면 잘 됐을 코드**입니다.
> 모델도, 데이터도, 학습 루프도 전부 같습니다 — **학습률 하나**만 틀렸을 뿐입니다.
>
> **기억할 것**: SGD 는 `0.1` 대, Adam 은 `1e-3` 대.
> 옵티마이저를 바꿀 때는 **lr 도 같이 바꿉니다.**

---

## 실습 3 — 스케줄러 적용 + 학습률 곡선 (12분)

```
   초반: 멀리 있으니 크게 가야 한다        lr 크게
   후반: 바닥 근처에서 크게 가면 튄다      lr 작게

        loss
         │＼
         │  ＼___              ← 초반: 성큼성큼
         │      ＼__
         │         ‾●‿●‿●     ← 후반: lr 이 그대로면 바닥에서 진동한다
         └────────────────── epoch
```

4주차에 *"lr 이 크면 발산, 작으면 못 간다"* 를 봤죠.
**정답은 "처음엔 크게, 나중엔 작게"** 입니다. 그걸 자동으로 해 주는 것이 **스케줄러**입니다.

| 스케줄러 | 동작 | 쓰임 |
|---|---|---|
| **`StepLR`** | 정해진 epoch마다 lr 에 `gamma` 를 곱한다 | 가장 단순 |
| **`CosineAnnealingLR`** | 코사인 곡선으로 부드럽게 0 쪽으로 | **요즘 기본** ★ 9·12주차에서 씀 |

> ⚠️ **함정 ★★**: `scheduler.step()` 을 **배치 안쪽에 두면** lr 이 배치 수만큼 빨리 줄어
> 학습이 멈춥니다. **epoch 루프 끝**에 둡니다. 매년 나오는 실수입니다.
> *(셀 3의 `train()` 안에서 이미 올바른 위치에 넣어 두었습니다 — 확인해 보세요.)*

In [ ]:
# 셀 7 — 학습 없이 스케줄러 모양만 미리 본다
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR

for label, make in [("StepLR(step=2, gamma=0.5)", lambda o: StepLR(o, step_size=2, gamma=0.5)),
                    ("Cosine(T_max=20)",          lambda o: CosineAnnealingLR(o, T_max=20))]:
    o = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.1)
    s = make(o)
    curve = []
    for _ in range(20):
        curve.append(o.param_groups[0]["lr"])
        o.step(); s.step()
    plt.plot(curve, marker=".", label=label)

plt.xlabel("epoch"); plt.ylabel("learning rate"); plt.legend()
plt.title("스케줄러 모양만 20 epoch 미리 보기"); plt.grid(alpha=0.3)
plt.show()

> **포인트**: 학습을 실행하기 전에 **스케줄러 모양만 먼저 그려 보는 것**은
> 실무에서 자주 쓰는 확인 방법입니다. 몇 epoch에 얼마나 줄어드는지 눈으로 보고 정합니다.

In [ ]:
# 셀 8 — 스케줄러 유무 비교 (학습까지)  ★ 시간이 좀 걸립니다
configs = {
    "스케줄러 없음": None,
    "StepLR":       lambda o: StepLR(o, step_size=2, gamma=0.5),
    "Cosine":       lambda o: CosineAnnealingLR(o, T_max=5),
}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, sched in configs.items():

    m, h, lrs = train("Momentum", epochs=5, sched=sched)

    ax[0].plot(range(1, 6), h,   marker="o", label=name)

    ax[1].plot(range(1, 6), lrs, marker="o", label=name)

    print(f"{name:12s} | 마지막 loss {h[-1]:.4f} | 정확도 {accuracy(m)*100:5.2f}%")

ax[0].set_title("손실"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set_title("학습률의 변화"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: 오른쪽 그림에서 **학습률이 실제로 줄어드는 모양**을 보세요.
> `StepLR` 은 계단, `Cosine` 은 부드러운 곡선입니다.
>
> 왼쪽 손실도 실제로 갈라집니다 (교수님 PC 실측) —
>
> ```
> 스케줄러 없음 | 마지막 loss 0.3376 | 정확도 84.37%
> StepLR       | 마지막 loss 0.2687 | 정확도 86.23%
> Cosine       | 마지막 loss 0.2525 | 정확도 86.59%
> ```
>
> 5 epoch 만으로도 **2%p 정도** 올라갔습니다. 다만 이 차이는 설정에 따라 작아지기도 합니다 —
> 스케줄러의 효과는 **긴 학습에서** 더 확실해집니다.
> 여기서 중요한 것은 *"스케줄러를 쓰면 무조건 좋아진다"* 가 아니라
> **학습률이 시간에 따라 변한다는 개념**과, 그것을 **곡선으로 확인하는 습관**입니다.
> 9주차 LoRA, 12주차 `Trainer` 에서 스케줄러가 기본으로 들어가 있습니다.

> **막히면**
>
> | 증상 | 원인 |
> |---|---|
> | 학습률이 안 변한다 | `scheduler.step()` 을 안 불렀다 |
> | 2 epoch 만에 lr 이 0 에 가까워진다 | `scheduler.step()` 이 **배치 루프 안**에 있다 ★ |
> | 손실이 발산 | Adam 에 `lr=0.1` 을 줬다 (실습 2) |

---

## 1교시 핵심 정리

| 개념 | 한 줄 정의 |
|------|-----------|
| **SGD → Momentum** ★ | 지그재그를 **관성**으로 상쇄 |
| **→ Adam** | **파라미터마다 학습률**을 다르게 자동 조절 |
| **→ AdamW** | 가중치 감쇠를 기울기에서 **분리**. 9·12주차 기본 |
| **lr 자릿수** ★ | SGD 는 `0.1` 대, Adam 은 `1e-3` 대. 바꾸면 **lr 도 바꾼다** |
| **스케줄러** | 처음엔 크게, 나중엔 작게. `StepLR`(계단) / `Cosine`(부드럽게) |
| **`scheduler.step()`** ★★ | **epoch 루프 끝**에. 배치 안에 두면 lr 이 너무 빨리 준다 |
| **공정한 비교** | 하나만 바꾸고 나머지(시드·epoch·batch)는 고정 |

### 반드시 외우기 ★

```python
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

for epoch in range(EPOCHS):
    for xb, yb in loader: ...
    sched.step()          # ★ epoch 끝에 한 번
```

### 1교시 체크리스트

- [ ] SGD → Momentum → Adam → AdamW 가 **각각 무엇을 해결했는지** 말할 수 있다 ★
- [ ] Adam 의 기본 lr 이 SGD 와 **자릿수가 다른** 것을 안다
- [ ] 같은 시드·같은 모델로 옵티마이저 3종을 비교했다
- [ ] **Adam 이 항상 이기지는 않는다**는 것을 결과로 확인했다
- [ ] Adam 에 `lr=0.1` 을 주면 발산하는 것을 직접 봤다 ★
- [ ] 스케줄러를 적용하고 **학습률 곡선**을 그렸다
- [ ] `scheduler.step()` 의 위치를 안다 ★★
- [ ] 비교 실험에 기록해야 할 조건 4가지를 안다

### 저장

`Kernel → Restart Kernel and Run All Cells` 로 다시 실행하고,
**출력이 살아 있는 상태로** `08_optimizers.ipynb` 로 저장합니다.

> **다음 시간 예고**: 정확도를 조금 올렸습니다. 그런데 **그게 진짜인지** 어떻게 알까요?
> 2교시에는 훈련 데이터를 일부러 1,000장으로 줄여서
> **훈련 곡선과 검증 곡선이 갈라지는 순간**을 봅니다. 그게 **과적합**입니다.